# FREIGHT-MNet — Improved M0 Tabular Baseline Notebook

This notebook implements the **M0 baseline stage** for FREIGHT-MNet. It is deliberately limited to non-graph tabular baselines so that the project has a clean benchmark before moving to MLP, HGT, or D-CQHGT.

The task is:

- Input unit: one `FAF origin × FAF destination × year` row.
- Labels: `truck_q25`, `truck_q50`, `truck_q75`.
- Temporal split: `2018–2022` train, `2023` validation, `2024` test.
- Main scope: `east_plus_gulf`.
- Robustness scope: `all`.

Implemented M0 baselines:

1. `global_train_median`
2. `od_historical_median`
3. `ridge`
4. `elasticnet`
5. `lightgbm_quantile` if LightGBM is available
6. `xgboost_quantile` or `xgboost_squarederror` if XGBoost is explicitly enabled

This notebook does **not** build `HeteroData`, does **not** use PyTorch Geometric, and does **not** train D-CQHGT. Its job is to create a reproducible baseline table for later comparison.


## 0. Why the previous `.py` run failed

The error you saw is an **environment binary compatibility problem**, not a modeling-code problem.

Your traceback says that a compiled package was built against **NumPy 1.x**, but your active environment has **NumPy 2.4.5**. PyCharm also loaded its plotting helper, which imported `matplotlib` before your script started. That triggered this failure:

```text
A module that was compiled using NumPy 1.x cannot be run in NumPy 2.4.5
AttributeError: _ARRAY_API not found
ImportError: cannot load module more than once per process
```

The notebook below avoids `matplotlib`, but NumPy, pandas, pyarrow, and scikit-learn still must be ABI-compatible. The safest solution is to create a clean M0 environment with NumPy `<2`.

Recommended Anaconda Prompt commands:

```bash
conda create -n freight_mnet_m0 python=3.11 -y
conda activate freight_mnet_m0
conda install -c conda-forge "numpy<2" pandas scikit-learn scipy pyarrow joblib jupyterlab ipykernel lightgbm xgboost -y
python -m ipykernel install --user --name freight_mnet_m0 --display-name "Python (freight_mnet_m0)"
```

Then open this notebook and select this kernel:

```text
Python (freight_mnet_m0)
```

Do not run the first test through PyCharm Scientific Mode until this environment problem is fixed. Use JupyterLab, VS Code Notebook, or a normal Jupyter Notebook kernel first.


In [1]:
# ============================================================
# Cell 0 — Environment preflight without importing NumPy first
# ============================================================
# This cell uses only Python standard-library modules before testing imports.
# It helps diagnose the NumPy ABI problem before the notebook starts training.

import sys
import subprocess
import platform
from importlib.metadata import version, PackageNotFoundError


def package_version(package_name: str) -> str:
    """Return the installed package version without importing the package."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not installed"


print("Python executable:", sys.executable)
print("Python version:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

for pkg in ["numpy", "pandas", "scikit-learn", "scipy", "pyarrow", "lightgbm", "xgboost", "matplotlib"]:
    print(f"{pkg:14s}: {package_version(pkg)}")

# Test core imports in a clean subprocess. If this fails, fix the environment
# before running the rest of the notebook.
preflight_code = r'''
import numpy as np
import pandas as pd
import sklearn
import pyarrow
print("SUBPROCESS_IMPORT_OK")
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)
print("pyarrow", pyarrow.__version__)
'''

completed = subprocess.run(
    [sys.executable, "-c", preflight_code],
    capture_output=True,
    text=True,
)

print("\nSubprocess return code:", completed.returncode)
print("--- subprocess stdout ---")
print(completed.stdout)
print("--- subprocess stderr ---")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "Environment preflight failed. Create the clean freight_mnet_m0 conda environment "
        "with numpy<2, then restart this notebook kernel."
    )


Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
numpy         : 2.4.5
pandas        : 2.3.3
scikit-learn  : 1.8.0
scipy         : 1.17.1
pyarrow       : 24.0.0
lightgbm      : 4.6.0
xgboost       : 3.2.0
matplotlib    : 3.10.9

Subprocess return code: 0
--- subprocess stdout ---
SUBPROCESS_IMPORT_OK
numpy 2.4.5
pandas 2.3.3
sklearn 1.8.0
pyarrow 24.0.0

--- subprocess stderr ---



## 1. Import libraries and check package versions

This notebook does **not** import `matplotlib`, because plotting is unnecessary for M0 baseline training and the previous crash was triggered through PyCharm's matplotlib helper.

If the next cell fails while importing NumPy, pandas, or scikit-learn, repair the environment first and restart the kernel.


In [2]:

from __future__ import annotations

import inspect
import json
import math
import os
import platform
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Tuple

# Import core numerical packages inside a guarded block so the error message is
# clearer than a raw binary-compatibility traceback.
try:
    import numpy as np
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "NumPy or pandas could not be imported. This is usually caused by a "
        "NumPy 2.x / compiled-extension mismatch. Install numpy==1.26.4 in a "
        "clean environment, restart the kernel, and run again."
    ) from exc

try:
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import ElasticNet, Ridge
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except Exception as exc:
    raise RuntimeError(
        "scikit-learn could not be imported. Reinstall scikit-learn after fixing "
        "NumPy, then restart the kernel."
    ) from exc

try:
    from joblib import dump as joblib_dump
except Exception:
    joblib_dump = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Executable:", sys.executable)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

import sklearn
print("scikit-learn:", sklearn.__version__)

try:
    import pyarrow as _pyarrow_check
    print("pyarrow:", _pyarrow_check.__version__)
except Exception as exc:
    print("pyarrow: unavailable or incompatible ->", repr(exc))

# Optional packages are checked here, but imported again inside their training
# functions. If they fail, the notebook can still run the safer baselines.
try:
    import lightgbm as _lgb_check
    print("LightGBM:", _lgb_check.__version__)
except Exception as exc:
    print("LightGBM: unavailable or incompatible ->", repr(exc))

try:
    import xgboost as _xgb_check
    print("XGBoost:", _xgb_check.__version__)
except Exception as exc:
    print("XGBoost: unavailable or incompatible ->", repr(exc))

Python: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
Executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
NumPy: 2.4.5
pandas: 2.3.3
scikit-learn: 1.8.0
pyarrow: 24.0.0
LightGBM: 4.6.0
XGBoost: 3.2.0


## 2. Experiment configuration

Edit only this cell for a normal run.

Important settings:

- `DATA_ROOT` must point to your project `Data` directory.
- `SCOPE` can be either `east_plus_gulf` or `all`.
- The recommended first run uses `SKIP_XGBOOST=True`, because XGBoost quantile support is version-sensitive.
- `OVERWRITE=True` removes the previous output folder for this run name and scope.

In [3]:

@dataclass(frozen=True)
class ExperimentConfig:
    """Central configuration for the M0 baseline experiment."""

    # Project data directory.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")

    # Dataset scope. Use "east_plus_gulf" for the main experiment and "all" for robustness.
    scope: str = "east_plus_gulf"

    # Output folder name under Data/10_experiments/<run_name>/<scope>/.
    run_name: str = "m0_baselines_v1_notebook"

    # Reproducibility and output behavior.
    seed: int = 42
    overwrite: bool = True
    save_models: bool = True

    # Evaluation behavior.
    # "clip_sort" clips negative predictions to zero and enforces q25 <= q50 <= q75.
    postprocess: str = "clip_sort"
    calibrate: bool = True
    use_sample_weight: bool = True

    # Optional model switches.
    skip_lightgbm: bool = False
    skip_xgboost: bool = True

    # Linear model hyperparameters.
    ridge_alpha: float = 10.0
    elasticnet_alpha: float = 0.001
    elasticnet_l1_ratio: float = 0.10
    elasticnet_max_iter: int = 20_000

    # LightGBM hyperparameters.
    lgb_n_estimators: int = 2_000
    lgb_learning_rate: float = 0.03
    lgb_num_leaves: int = 63
    lgb_min_child_samples: int = 50
    lgb_early_stopping_rounds: int = 100

    # XGBoost hyperparameters.
    xgb_n_estimators: int = 900
    xgb_learning_rate: float = 0.03
    xgb_max_depth: int = 6
    xgb_tree_method: str = "hist"

    # Parallelism. -1 means use all available CPU cores where supported.
    n_jobs: int = -1


CONFIG = ExperimentConfig()

# Validate a few key config values early.
assert CONFIG.scope in {"east_plus_gulf", "all"}, "scope must be 'east_plus_gulf' or 'all'."
assert CONFIG.postprocess in {"none", "clip", "sort", "clip_sort"}, "Invalid postprocess mode."

print(CONFIG)

ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='m0_baselines_v1_notebook', seed=42, overwrite=True, save_models=True, postprocess='clip_sort', calibrate=True, use_sample_weight=True, skip_lightgbm=False, skip_xgboost=True, ridge_alpha=10.0, elasticnet_alpha=0.001, elasticnet_l1_ratio=0.1, elasticnet_max_iter=20000, lgb_n_estimators=2000, lgb_learning_rate=0.03, lgb_num_leaves=63, lgb_min_child_samples=50, lgb_early_stopping_rounds=100, xgb_n_estimators=900, xgb_learning_rate=0.03, xgb_max_depth=6, xgb_tree_method='hist', n_jobs=-1)


## 3. Path management

This cell resolves all required input and output paths.

Required input files:

```text
Data/08_processed/model_ready/freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Data/08_processed/model_ready/freight_mnet_supervised_edges_2018_2024_all.parquet
Data/08_processed/model_ready/_metadata/freight_mnet_supervised_feature_manifest.json
```

The output folder is:

```text
Data/10_experiments/m0_baselines_v1_notebook/<scope>/
```

In [4]:

@dataclass(frozen=True)
class ExperimentPaths:
    """Resolved filesystem paths used by the experiment."""

    data_root: Path
    scope: str
    model_ready_dir: Path
    supervised_path: Path
    manifest_path: Path
    output_dir: Path
    models_dir: Path


def resolve_paths(cfg: ExperimentConfig) -> ExperimentPaths:
    """Resolve all input and output paths from the experiment config."""
    data_root = cfg.data_root.expanduser().resolve()
    model_ready_dir = data_root / "08_processed" / "model_ready"

    supervised_file = (
        "freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet"
        if cfg.scope == "east_plus_gulf"
        else "freight_mnet_supervised_edges_2018_2024_all.parquet"
    )

    supervised_path = model_ready_dir / supervised_file
    manifest_path = model_ready_dir / "_metadata" / "freight_mnet_supervised_feature_manifest.json"
    output_dir = data_root / "10_experiments" / cfg.run_name / cfg.scope
    models_dir = output_dir / "models"

    return ExperimentPaths(
        data_root=data_root,
        scope=cfg.scope,
        model_ready_dir=model_ready_dir,
        supervised_path=supervised_path,
        manifest_path=manifest_path,
        output_dir=output_dir,
        models_dir=models_dir,
    )


def prepare_output_dir(paths: ExperimentPaths, overwrite: bool) -> None:
    """Create the output directory, optionally deleting previous results."""
    if paths.output_dir.exists() and overwrite:
        shutil.rmtree(paths.output_dir)
    paths.output_dir.mkdir(parents=True, exist_ok=True)
    paths.models_dir.mkdir(parents=True, exist_ok=True)


PATHS = resolve_paths(CONFIG)

print("Data root:       ", PATHS.data_root)
print("Supervised table:", PATHS.supervised_path)
print("Feature manifest:", PATHS.manifest_path)
print("Output folder:   ", PATHS.output_dir)

Data root:        E:\NetworkOptimization\pythonProject1\Data
Supervised table: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Output folder:    E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf


## 4. Constants and leakage guards

The feature manifest should contain only predictive features. The following columns must **not** enter the model feature matrix:

- label columns such as `truck_q25`, `truck_q50`, `truck_q75`
- label-derived intervals/gaps such as `truck_iqr`
- FMI aggregation diagnostics such as `n_fmi_county_pairs` and `obs_weight_sum`
- raw input quantile aggregation fields such as `input_q50_min`

Some diagnostic columns can still be used for **loss weighting** or **stress/sparse evaluation**, but not as predictors.

In [5]:

TAUS = np.asarray([0.25, 0.50, 0.75], dtype=np.float64)
DEFAULT_LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
ID_COLUMNS = ["scope", "faf_orig", "faf_dest", "faf_od", "year", "split"]

# These columns are either labels, label-derived quantities, or FMI aggregation
# diagnostics. They are allowed for weighting and diagnostics only.
LEAKAGE_COLUMNS = {
    "truck_q25",
    "truck_q50",
    "truck_q75",
    "truck_iqr",
    "truck_q75_q50_gap",
    "truck_q50_q25_gap",
    "truck_iqr_over_q50",
    "n_fmi_county_pairs",
    "obs_weight_sum",
    "obs_weight_mean",
    "obs_weight_max",
    "input_q25_weighted_mean",
    "input_q50_weighted_mean",
    "input_q75_weighted_mean",
    "input_q25_min",
    "input_q50_min",
    "input_q75_min",
    "input_q25_max",
    "input_q50_max",
    "input_q75_max",
    "n_orig_counties",
    "n_dest_counties",
}

# Conservative substring checks. These are intentionally focused on target and
# FMI aggregation diagnostics. They do not block valid features like county
# counts or zone metadata.
SUSPICIOUS_FEATURE_TOKENS = [
    "truck_q",
    "input_q",
    "obs_weight",
    "fmi_county_pairs",
    "weighted_mean_q",
]

## 5. Data loading utilities

These functions read the model-ready table and the feature manifest, validate the temporal split, and create train/validation/test arrays.

Design choices:

- The notebook reads `feature_columns` from the manifest instead of inferring features from the table.
- Non-numeric manifest columns are dropped for M0 numeric tabular baselines.
- Sample weights are computed from `obs_weight_sum` if available, otherwise from `n_fmi_county_pairs`.
- Imputation and scaling are fitted only inside model pipelines on the training split.

In [6]:

@dataclass
class DatasetBundle:
    """Container for the loaded supervised data and train/val/test matrices."""

    df: pd.DataFrame
    feature_columns: List[str]
    label_columns: List[str]
    weight_columns: List[str]
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame
    w_train: np.ndarray
    w_val: np.ndarray
    w_test: np.ndarray


def read_json(path: Path) -> Dict[str, Any]:
    """Read a UTF-8 JSON file."""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_faf_code(series: pd.Series) -> pd.Series:
    """Normalize FAF zone codes to zero-padded three-character strings."""
    out = series.astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.str.replace(r"[^0-9]", "", regex=True)
    return out.str.zfill(3)


def ensure_required_files(paths: ExperimentPaths) -> None:
    """Raise a clear error if required model-ready files are missing."""
    missing = [str(p) for p in [paths.supervised_path, paths.manifest_path] if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required input file(s):\n" + "\n".join(missing))


def create_or_validate_temporal_split(df: pd.DataFrame) -> pd.DataFrame:
    """Validate the official temporal split or create it from year if absent."""
    out = df.copy()

    if "year" not in out.columns:
        raise RuntimeError("The supervised table must contain a 'year' column.")

    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out = out.dropna(subset=["year"]).copy()
    out["year"] = out["year"].astype(int)

    if "split" not in out.columns:
        # Safe fallback for model-ready tables that contain years but not a split column.
        out["split"] = np.select(
            [out["year"].between(2018, 2022), out["year"].eq(2023), out["year"].eq(2024)],
            ["train", "val", "test"],
            default="unused",
        )
    else:
        out["split"] = out["split"].astype(str).str.lower().str.strip()

    out = out[out["split"].isin(["train", "val", "test"])].copy()

    split_years = (
        out.groupby("split")["year"]
        .apply(lambda s: sorted(int(x) for x in pd.Series(s).dropna().unique()))
        .to_dict()
    )
    expected = {"train": [2018, 2019, 2020, 2021, 2022], "val": [2023], "test": [2024]}

    for split_name, years in expected.items():
        if split_name not in split_years:
            raise RuntimeError(f"Missing split={split_name}. Found split years: {split_years}")
        if split_years[split_name] != years:
            raise RuntimeError(
                f"Unexpected years for split={split_name}: {split_years[split_name]}. "
                f"Expected {years}."
            )

    return out


def select_feature_columns(df: pd.DataFrame, manifest: Mapping[str, Any]) -> List[str]:
    """Select numeric manifest features and enforce leakage checks."""
    if "feature_columns" not in manifest:
        raise RuntimeError("Feature manifest missing key: 'feature_columns'.")

    manifest_features = list(manifest["feature_columns"])
    missing = [c for c in manifest_features if c not in df.columns]
    if missing:
        preview = "\n".join(missing[:50])
        suffix = "\n..." if len(missing) > 50 else ""
        raise RuntimeError("The supervised table is missing manifest features:\n" + preview + suffix)

    numeric_features: List[str] = []
    dropped_non_numeric: List[str] = []

    for col in manifest_features:
        if col in LEAKAGE_COLUMNS or col in ID_COLUMNS:
            raise RuntimeError(f"Leakage or ID column found in feature manifest: {col}")

        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
            continue

        # Some parquet readers may return numeric-looking columns as object/string.
        # Accept such columns only if most non-missing values can be converted safely.
        converted = pd.to_numeric(df[col], errors="coerce")
        original_non_missing = int(df[col].notna().sum())
        converted_non_missing = int(converted.notna().sum())
        if original_non_missing > 0 and converted_non_missing / original_non_missing >= 0.95:
            numeric_features.append(col)
        else:
            dropped_non_numeric.append(col)

    suspicious = []
    for col in numeric_features:
        low = col.lower()
        if any(token in low for token in SUSPICIOUS_FEATURE_TOKENS):
            suspicious.append(col)

    if suspicious:
        raise RuntimeError(
            "Potential leakage columns detected in feature_columns. Remove them before training:\n"
            + "\n".join(suspicious)
        )

    if dropped_non_numeric:
        print("Non-numeric manifest features dropped for M0 numeric baselines:")
        for col in dropped_non_numeric:
            print("  -", col)

    if not numeric_features:
        raise RuntimeError("No numeric feature columns found after leakage checks.")

    return sorted(numeric_features)


def make_design_matrix(df: pd.DataFrame, feature_columns: List[str]) -> pd.DataFrame:
    """Create a numeric design matrix with infinities converted to missing values."""
    X = df[feature_columns].copy()
    for col in feature_columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.replace([np.inf, -np.inf], np.nan)


def make_sample_weights(df: pd.DataFrame, use_sample_weight: bool) -> np.ndarray:
    """Create normalized sample weights from diagnostics, without using them as features."""
    if not use_sample_weight:
        return np.ones(len(df), dtype=np.float64)

    if "obs_weight_sum" in df.columns:
        raw = pd.to_numeric(df["obs_weight_sum"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
        raw = np.maximum(raw, 0.0)
        weights = np.log1p(raw)
    elif "n_fmi_county_pairs" in df.columns:
        raw = pd.to_numeric(df["n_fmi_county_pairs"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
        raw = np.maximum(raw, 0.0)
        weights = np.sqrt(np.maximum(raw, 1.0))
    else:
        return np.ones(len(df), dtype=np.float64)

    if not np.isfinite(weights).all() or float(np.nanmean(weights)) <= 0:
        return np.ones(len(df), dtype=np.float64)

    weights = np.where(np.isfinite(weights), weights, 1.0)
    weights = weights / np.mean(weights)
    return weights.astype(np.float64)


def load_dataset(paths: ExperimentPaths, cfg: ExperimentConfig) -> DatasetBundle:
    """Load the model-ready supervised table and return train/val/test arrays."""
    ensure_required_files(paths)
    manifest = read_json(paths.manifest_path)
    df = pd.read_parquet(paths.supervised_path)

    # Normalize common OD identifier fields.
    for col in ["faf_orig", "faf_dest"]:
        if col in df.columns:
            df[col] = normalize_faf_code(df[col])

    if "faf_od" not in df.columns and {"faf_orig", "faf_dest"}.issubset(df.columns):
        df["faf_od"] = df["faf_orig"].astype(str) + "->" + df["faf_dest"].astype(str)

    label_columns = list(manifest.get("label_columns", DEFAULT_LABEL_COLUMNS))
    missing_labels = [col for col in label_columns if col not in df.columns]
    if missing_labels:
        raise RuntimeError(f"Missing label columns: {missing_labels}")

    for col in label_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=label_columns).copy()
    df = create_or_validate_temporal_split(df)

    # Target quantiles must be monotone before training.
    is_monotone = (df[label_columns[0]] <= df[label_columns[1]]).all() and (df[label_columns[1]] <= df[label_columns[2]]).all()
    if not is_monotone:
        raise RuntimeError("Target quantiles are not monotone. Re-check the DataPreprocess output.")

    feature_columns = select_feature_columns(df, manifest)
    X = make_design_matrix(df, feature_columns)

    train_mask = df["split"].eq("train")
    val_mask = df["split"].eq("val")
    test_mask = df["split"].eq("test")

    train_df = df.loc[train_mask].reset_index(drop=True)
    val_df = df.loc[val_mask].reset_index(drop=True)
    test_df = df.loc[test_mask].reset_index(drop=True)

    bundle = DatasetBundle(
        df=df.reset_index(drop=True),
        feature_columns=feature_columns,
        label_columns=label_columns,
        weight_columns=list(manifest.get("loss_weight_columns", [])),
        X_train=X.loc[train_mask].reset_index(drop=True),
        X_val=X.loc[val_mask].reset_index(drop=True),
        X_test=X.loc[test_mask].reset_index(drop=True),
        y_train=train_df[label_columns].to_numpy(dtype=np.float64),
        y_val=val_df[label_columns].to_numpy(dtype=np.float64),
        y_test=test_df[label_columns].to_numpy(dtype=np.float64),
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        w_train=make_sample_weights(train_df, cfg.use_sample_weight),
        w_val=make_sample_weights(val_df, cfg.use_sample_weight),
        w_test=make_sample_weights(test_df, cfg.use_sample_weight),
    )

    return bundle

## 6. Load and inspect the supervised table

This cell performs the first real data check. It should print:

- total row count
- number of selected numeric features
- train/validation/test counts
- year counts
- label medians in the training split
- sample-weight summary

If this cell fails, fix the data path or the feature manifest before training any model.

In [7]:

prepare_output_dir(PATHS, overwrite=CONFIG.overwrite)
BUNDLE = load_dataset(PATHS, CONFIG)

summary = {
    "scope": PATHS.scope,
    "rows_total": len(BUNDLE.df),
    "n_features": len(BUNDLE.feature_columns),
    "split_counts": BUNDLE.df["split"].value_counts().to_dict(),
    "year_counts": BUNDLE.df["year"].value_counts().sort_index().to_dict(),
    "unique_faf_od_pairs": int(BUNDLE.df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
    if {"faf_orig", "faf_dest"}.issubset(BUNDLE.df.columns)
    else None,
    "train_label_medians": dict(zip(BUNDLE.label_columns, np.median(BUNDLE.y_train, axis=0).round(3))),
    "train_weight_mean": float(np.mean(BUNDLE.w_train)),
    "train_weight_min": float(np.min(BUNDLE.w_train)),
    "train_weight_max": float(np.max(BUNDLE.w_train)),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

# Show a compact feature preview without printing all 64+ columns.
print("\nFirst 20 selected feature columns:")
for col in BUNDLE.feature_columns[:20]:
    print("  -", col)

print("\nData preview:")
display(BUNDLE.df.head())

{
  "scope": "east_plus_gulf",
  "rows_total": 73972,
  "n_features": 64,
  "split_counts": {
    "train": 52773,
    "val": 10625,
    "test": 10574
  },
  "year_counts": {
    "2018": 9936,
    "2019": 10704,
    "2020": 10761,
    "2021": 10721,
    "2022": 10651,
    "2023": 10625,
    "2024": 10574
  },
  "unique_faf_od_pairs": 10762,
  "train_label_medians": {
    "truck_q25": 1494.11,
    "truck_q50": 2314.45,
    "truck_q75": 3631.97
  },
  "train_weight_mean": 1.0,
  "train_weight_min": 0.1246731052328745,
  "train_weight_max": 2.099924925946007
}

First 20 selected feature columns:
  - dest_east_plus_gulf_county_share
  - dest_east_plus_gulf_faf_flag_any_county
  - dest_max_county_centroid_lon
  - dest_min_county_centroid_lon
  - dest_n_counties
  - dest_n_east_plus_gulf_counties
  - has_multimodal_demand
  - has_rail_demand
  - has_truck_demand
  - log1p_tmiles_multimodal
  - log1p_tmiles_rail
  - log1p_tmiles_truck
  - log1p_tons_multimodal
  - log1p_tons_rail
  - log1p_ton

,scope,faf_orig,faf_dest,faf_od,year,split,truck_q25,truck_q50,truck_q75,truck_iqr,...,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties,truck_q75_q50_gap,truck_q50_q25_gap,scope_from_label
0,east_plus_gulf,011,011,011->011,2018,train,1.00,37.02,71.28,70.28,...,33.156860,47.766945,155.222758,1.00,562.50,11,11,34.26,36.02,east_plus_gulf
1,east_plus_gulf,011,012,011->012,2018,train,248.18,943.27,1925.53,1677.35,...,516.764447,966.672508,2184.157713,173.83,1979.45,11,2,982.26,695.09,east_plus_gulf
2,east_plus_gulf,011,019,011->019,2018,train,48.00,100.10,218.95,170.95,...,95.525005,178.619190,492.856903,1.00,1646.70,11,54,118.85,52.10,east_plus_gulf
3,east_plus_gulf,011,050,011->050,2018,train,404.01,1010.39,1818.68,1414.67,...,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63,808.29,606.38,east_plus_gulf
4,east_plus_gulf,011,091,011->091,2018,train,2579.38,3718.82,5127.33,2547.95,...,2510.728667,3559.798000,5716.874000,2557.95,4924.82,8,2,1408.51,1139.44,east_plus_gulf


## 7. Metric and prediction utilities

The primary metrics are:

- `mae_q25`, `mae_q50`, `mae_q75`
- `rmse_q50`
- `pinball_q25`, `pinball_q50`, `pinball_q75`, `pinball_mean`
- `iqr_mae`
- stress metrics on the top 10% realized `q75` and top 10% realized IQR
- sparse-label metrics on the lowest 25% `n_fmi_county_pairs`, if available

The notebook also records raw quantile crossing before post-processing. This is important because non-monotone baselines can output `q25 > q50` or `q50 > q75`.

In [8]:

def pinball_loss_array(y_true: np.ndarray, y_pred: np.ndarray, taus: np.ndarray = TAUS) -> np.ndarray:
    """Return elementwise pinball loss for q25, q50, and q75."""
    diff = y_true - y_pred
    return np.maximum(taus.reshape(1, -1) * diff, (taus.reshape(1, -1) - 1.0) * diff)


def weighted_mean(values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Compute a finite weighted mean with safe fallbacks."""
    values = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(values)

    if weights is None:
        return float(np.mean(values[valid])) if valid.any() else float("nan")

    weights = np.asarray(weights, dtype=np.float64)
    valid = valid & np.isfinite(weights) & (weights >= 0)
    if not valid.any() or float(np.sum(weights[valid])) <= 0:
        return float("nan")

    return float(np.average(values[valid], weights=weights[valid]))


def crossing_rate(pred: np.ndarray) -> float:
    """Return the fraction of rows with q25 > q50 or q50 > q75."""
    if len(pred) == 0:
        return float("nan")
    bad = (pred[:, 0] > pred[:, 1]) | (pred[:, 1] > pred[:, 2])
    return float(np.mean(bad))


def negative_rate(pred: np.ndarray) -> float:
    """Return the fraction of predicted quantile values below zero."""
    if len(pred) == 0:
        return float("nan")
    return float(np.mean(pred < 0))


def postprocess_predictions(pred: np.ndarray, mode: str) -> np.ndarray:
    """Apply nonnegativity and/or monotonicity post-processing to predictions."""
    out = np.asarray(pred, dtype=np.float64).copy()

    if mode in {"clip", "clip_sort"}:
        out = np.maximum(out, 0.0)
    if mode in {"sort", "clip_sort"}:
        out = np.sort(out, axis=1)

    return out


def validation_median_calibration(y_val: np.ndarray, pred_val: np.ndarray) -> np.ndarray:
    """Compute global median residual correction on the validation split."""
    residual = y_val - pred_val
    return np.nanmedian(residual, axis=0)


def apply_calibration(pred: np.ndarray, correction: np.ndarray, postprocess: str) -> np.ndarray:
    """Apply validation residual correction and re-run post-processing."""
    corrected = pred + correction.reshape(1, -1)
    return postprocess_predictions(corrected, postprocess)


def evaluate_predictions(
    split_df: pd.DataFrame,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_pred_raw: Optional[np.ndarray] = None,
    sample_weight: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Evaluate prediction arrays for one split."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    abs_err = np.abs(y_true - y_pred)
    sq_err = (y_true - y_pred) ** 2
    pinball = pinball_loss_array(y_true, y_pred, TAUS)

    iqr_true = y_true[:, 2] - y_true[:, 0]
    iqr_pred = y_pred[:, 2] - y_pred[:, 0]
    iqr_err = np.abs(iqr_true - iqr_pred)

    metrics: Dict[str, Any] = {
        "n_rows": int(len(y_true)),
        "mae_q25": float(np.mean(abs_err[:, 0])),
        "mae_q50": float(np.mean(abs_err[:, 1])),
        "mae_q75": float(np.mean(abs_err[:, 2])),
        "rmse_q50": float(math.sqrt(np.mean(sq_err[:, 1]))),
        "pinball_q25": float(np.mean(pinball[:, 0])),
        "pinball_q50": float(np.mean(pinball[:, 1])),
        "pinball_q75": float(np.mean(pinball[:, 2])),
        "pinball_mean": float(np.mean(pinball)),
        "iqr_mae": float(np.mean(iqr_err)),
        "bias_q25": float(np.mean(y_pred[:, 0] - y_true[:, 0])),
        "bias_q50": float(np.mean(y_pred[:, 1] - y_true[:, 1])),
        "bias_q75": float(np.mean(y_pred[:, 2] - y_true[:, 2])),
        "pred_crossing_rate": crossing_rate(y_pred),
        "pred_negative_rate": negative_rate(y_pred),
        "q50_inside_pred_q25_q75_rate": float(np.mean((y_true[:, 1] >= y_pred[:, 0]) & (y_true[:, 1] <= y_pred[:, 2]))),
    }

    if y_pred_raw is not None:
        metrics["raw_crossing_rate"] = crossing_rate(y_pred_raw)
        metrics["raw_negative_rate"] = negative_rate(y_pred_raw)

    if sample_weight is not None:
        metrics["weighted_pinball_mean"] = weighted_mean(np.mean(pinball, axis=1), sample_weight)
        metrics["weighted_mae_q50"] = weighted_mean(abs_err[:, 1], sample_weight)
        metrics["weighted_mae_q75"] = weighted_mean(abs_err[:, 2], sample_weight)

    # Stress split: top 10% by realized q75 in the current evaluation split.
    if len(y_true) >= 10:
        q75_threshold = float(np.quantile(y_true[:, 2], 0.90))
        q75_stress = y_true[:, 2] >= q75_threshold
        if q75_stress.any():
            metrics["stress_top10_true_q75_threshold"] = q75_threshold
            metrics["stress_top10_true_q75_n"] = int(q75_stress.sum())
            metrics["stress_top10_mae_q75"] = float(np.mean(abs_err[q75_stress, 2]))
            metrics["stress_top10_pinball_q75"] = float(np.mean(pinball[q75_stress, 2]))

        iqr_threshold = float(np.quantile(iqr_true, 0.90))
        iqr_stress = iqr_true >= iqr_threshold
        if iqr_stress.any():
            metrics["stress_top10_true_iqr_threshold"] = iqr_threshold
            metrics["stress_top10_iqr_n"] = int(iqr_stress.sum())
            metrics["stress_top10_iqr_mae"] = float(np.mean(iqr_err[iqr_stress]))
            metrics["stress_top10_iqr_mae_q75"] = float(np.mean(abs_err[iqr_stress, 2]))

    # Sparse-label split: bottom 25% by FMI county-pair support, if available.
    if "n_fmi_county_pairs" in split_df.columns and len(split_df) >= 10:
        support = pd.to_numeric(split_df["n_fmi_county_pairs"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
        sparse_threshold = float(np.quantile(support, 0.25))
        sparse = support <= sparse_threshold
        if sparse.any():
            metrics["sparse_bottom25_county_pairs_threshold"] = sparse_threshold
            metrics["sparse_bottom25_n"] = int(sparse.sum())
            metrics["sparse_bottom25_mae_q50"] = float(np.mean(abs_err[sparse, 1]))
            metrics["sparse_bottom25_mae_q75"] = float(np.mean(abs_err[sparse, 2]))
            metrics["sparse_bottom25_pinball_mean"] = float(np.mean(pinball[sparse]))

    return metrics


def prediction_frame(
    split_df: pd.DataFrame,
    model_name: str,
    split_name: str,
    y_pred: np.ndarray,
    y_pred_raw: np.ndarray,
    label_columns: List[str],
    variant: str,
) -> pd.DataFrame:
    """Create a long artifact table with predictions and absolute errors."""
    keep_columns = [col for col in ID_COLUMNS if col in split_df.columns]
    diagnostic_columns = [col for col in ["n_fmi_county_pairs", "obs_weight_sum"] if col in split_df.columns]

    out = split_df[keep_columns + diagnostic_columns].copy()
    out.insert(0, "model", model_name)
    out.insert(1, "variant", variant)
    out.insert(2, "eval_split", split_name)

    for j, label in enumerate(label_columns):
        out[f"actual_{label}"] = split_df[label].to_numpy(dtype=np.float64)
        out[f"pred_{label}"] = y_pred[:, j]
        out[f"pred_raw_{label}"] = y_pred_raw[:, j]
        out[f"abs_err_{label}"] = np.abs(out[f"actual_{label}"] - out[f"pred_{label}"])

    out["actual_iqr"] = out[f"actual_{label_columns[2]}"] - out[f"actual_{label_columns[0]}"]
    out["pred_iqr"] = out[f"pred_{label_columns[2]}"] - out[f"pred_{label_columns[0]}"]
    return out

## 8. Baseline model implementations

The first two baselines do not use any feature columns:

- `global_train_median`: predicts the training median quantiles for every row.
- `od_historical_median`: predicts the historical training median for the same FAF OD pair, falling back to global medians for unseen pairs.

The linear baselines use scikit-learn pipelines:

- `SimpleImputer(strategy="median")`
- `StandardScaler()`
- `Ridge` or `ElasticNet`

The tree baselines are optional:

- LightGBM uses separate quantile regressors for `tau=0.25`, `0.50`, and `0.75`.
- XGBoost first tries `reg:quantileerror`; if unsupported, it falls back to `reg:squarederror` for each target.

In [9]:

def fit_global_train_median(bundle: DatasetBundle, cfg: ExperimentConfig) -> Dict[str, Any]:
    """Fit the global training median baseline."""
    del cfg  # This baseline does not need configuration values.
    median = np.median(bundle.y_train, axis=0)

    return {
        "name": "global_train_median",
        "model": {"train_median": median.tolist()},
        "pred_train": np.tile(median.reshape(1, -1), (len(bundle.train_df), 1)),
        "pred_val": np.tile(median.reshape(1, -1), (len(bundle.val_df), 1)),
        "pred_test": np.tile(median.reshape(1, -1), (len(bundle.test_df), 1)),
    }


def fit_od_historical_median(bundle: DatasetBundle, cfg: ExperimentConfig) -> Dict[str, Any]:
    """Fit the OD-level historical median baseline using train years only."""
    del cfg
    if "faf_od" not in bundle.train_df.columns:
        raise RuntimeError("The 'faf_od' column is required for the OD historical median baseline.")

    od_medians = bundle.train_df.groupby("faf_od")[bundle.label_columns].median()
    global_median = np.median(bundle.y_train, axis=0)

    def predict(df: pd.DataFrame) -> np.ndarray:
        pred = np.tile(global_median.reshape(1, -1), (len(df), 1))
        keys = df["faf_od"].astype(str).to_numpy()

        # A simple loop is clear and fast enough at this dataset size.
        for i, key in enumerate(keys):
            if key in od_medians.index:
                pred[i, :] = od_medians.loc[key, bundle.label_columns].to_numpy(dtype=np.float64)

        return pred

    return {
        "name": "od_historical_median",
        "model": {"n_train_od_pairs": int(len(od_medians)), "global_train_median": global_median.tolist()},
        "pred_train": predict(bundle.train_df),
        "pred_val": predict(bundle.val_df),
        "pred_test": predict(bundle.test_df),
    }


def fit_ridge(bundle: DatasetBundle, cfg: ExperimentConfig) -> Dict[str, Any]:
    """Fit a multi-output Ridge regression baseline."""
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=cfg.ridge_alpha, random_state=cfg.seed)),
        ]
    )

    fit_kwargs: Dict[str, Any] = {}
    if cfg.use_sample_weight:
        fit_kwargs["ridge__sample_weight"] = bundle.w_train

    model.fit(bundle.X_train, bundle.y_train, **fit_kwargs)

    return {
        "name": "ridge",
        "model": model,
        "pred_train": model.predict(bundle.X_train),
        "pred_val": model.predict(bundle.X_val),
        "pred_test": model.predict(bundle.X_test),
    }


def fit_elasticnet(bundle: DatasetBundle, cfg: ExperimentConfig) -> Dict[str, Any]:
    """Fit one ElasticNet model per target quantile."""
    base_model = ElasticNet(
        alpha=cfg.elasticnet_alpha,
        l1_ratio=cfg.elasticnet_l1_ratio,
        max_iter=cfg.elasticnet_max_iter,
        random_state=cfg.seed,
        selection="cyclic",
    )

    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("elasticnet", MultiOutputRegressor(base_model, n_jobs=cfg.n_jobs)),
        ]
    )

    # Keep ElasticNet unweighted for broad scikit-learn compatibility.
    model.fit(bundle.X_train, bundle.y_train)

    return {
        "name": "elasticnet",
        "model": model,
        "pred_train": model.predict(bundle.X_train),
        "pred_val": model.predict(bundle.X_val),
        "pred_test": model.predict(bundle.X_test),
    }

In [10]:

def fit_lightgbm_quantile(bundle: DatasetBundle, cfg: ExperimentConfig) -> Optional[Dict[str, Any]]:
    """Fit separate LightGBM quantile regressors for q25, q50, and q75."""
    if cfg.skip_lightgbm:
        print("[SKIP] LightGBM disabled by config.skip_lightgbm=True")
        return None

    try:
        import lightgbm as lgb
    except Exception as exc:
        print("[SKIP] LightGBM is unavailable or incompatible:", repr(exc))
        return None

    models: List[Any] = []
    pred_train = np.zeros_like(bundle.y_train, dtype=np.float64)
    pred_val = np.zeros_like(bundle.y_val, dtype=np.float64)
    pred_test = np.zeros_like(bundle.y_test, dtype=np.float64)

    for j, tau in enumerate(TAUS):
        print(f"[LightGBM] Training quantile tau={tau:.2f} for target={bundle.label_columns[j]}")

        model = lgb.LGBMRegressor(
            objective="quantile",
            alpha=float(tau),
            n_estimators=cfg.lgb_n_estimators,
            learning_rate=cfg.lgb_learning_rate,
            num_leaves=cfg.lgb_num_leaves,
            min_child_samples=cfg.lgb_min_child_samples,
            subsample=0.90,
            subsample_freq=1,
            colsample_bytree=0.90,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=cfg.seed + j,
            n_jobs=cfg.n_jobs,
            verbosity=-1,
        )

        fit_kwargs: Dict[str, Any] = {
            "X": bundle.X_train,
            "y": bundle.y_train[:, j],
        }

        if cfg.use_sample_weight:
            fit_kwargs["sample_weight"] = bundle.w_train

        # Use validation early stopping when the installed LightGBM wrapper supports it.
        try:
            callbacks = [lgb.early_stopping(cfg.lgb_early_stopping_rounds, verbose=False)]
            fit_kwargs.update({"eval_set": [(bundle.X_val, bundle.y_val[:, j])], "callbacks": callbacks})
            if cfg.use_sample_weight:
                fit_kwargs["eval_sample_weight"] = [bundle.w_val]
            model.fit(**fit_kwargs)
        except TypeError:
            # Older wrappers may not accept callbacks or eval_sample_weight.
            fit_kwargs.pop("eval_set", None)
            fit_kwargs.pop("callbacks", None)
            fit_kwargs.pop("eval_sample_weight", None)
            model.fit(**fit_kwargs)

        models.append(model)
        pred_train[:, j] = model.predict(bundle.X_train)
        pred_val[:, j] = model.predict(bundle.X_val)
        pred_test[:, j] = model.predict(bundle.X_test)

    return {
        "name": "lightgbm_quantile",
        "model": models,
        "pred_train": pred_train,
        "pred_val": pred_val,
        "pred_test": pred_test,
    }


def fit_xgboost(bundle: DatasetBundle, cfg: ExperimentConfig) -> Optional[Dict[str, Any]]:
    """Fit XGBoost quantile models if supported, otherwise square-error models."""
    if cfg.skip_xgboost:
        print("[SKIP] XGBoost disabled by config.skip_xgboost=True")
        return None

    try:
        import xgboost as xgb
    except Exception as exc:
        print("[SKIP] XGBoost is unavailable or incompatible:", repr(exc))
        return None

    def train_with_objective(objective: str, use_quantile_alpha: bool) -> Tuple[List[Any], np.ndarray, np.ndarray, np.ndarray]:
        models: List[Any] = []
        pred_train = np.zeros_like(bundle.y_train, dtype=np.float64)
        pred_val = np.zeros_like(bundle.y_val, dtype=np.float64)
        pred_test = np.zeros_like(bundle.y_test, dtype=np.float64)

        for j, tau in enumerate(TAUS):
            print(f"[XGBoost] Training target={bundle.label_columns[j]} objective={objective} tau={tau:.2f}")

            params: Dict[str, Any] = {
                "objective": objective,
                "n_estimators": cfg.xgb_n_estimators,
                "learning_rate": cfg.xgb_learning_rate,
                "max_depth": cfg.xgb_max_depth,
                "subsample": 0.90,
                "colsample_bytree": 0.90,
                "reg_lambda": 1.0,
                "random_state": cfg.seed + j,
                "n_jobs": cfg.n_jobs,
                "tree_method": cfg.xgb_tree_method,
                "verbosity": 0,
            }

            if use_quantile_alpha:
                params["quantile_alpha"] = float(tau)
                params["eval_metric"] = "quantile"
            else:
                params["eval_metric"] = "rmse"

            model = xgb.XGBRegressor(**params)
            fit_kwargs: Dict[str, Any] = {}
            if cfg.use_sample_weight:
                fit_kwargs["sample_weight"] = bundle.w_train

            model.fit(bundle.X_train, bundle.y_train[:, j], **fit_kwargs)
            models.append(model)
            pred_train[:, j] = model.predict(bundle.X_train)
            pred_val[:, j] = model.predict(bundle.X_val)
            pred_test[:, j] = model.predict(bundle.X_test)

        return models, pred_train, pred_val, pred_test

    try:
        models, pred_train, pred_val, pred_test = train_with_objective("reg:quantileerror", use_quantile_alpha=True)
        return {
            "name": "xgboost_quantile",
            "model": models,
            "pred_train": pred_train,
            "pred_val": pred_val,
            "pred_test": pred_test,
        }
    except Exception as quantile_exc:
        print("[WARN] XGBoost quantile objective failed. Falling back to reg:squarederror.")
        print("       Quantile failure:", repr(quantile_exc))

    try:
        models, pred_train, pred_val, pred_test = train_with_objective("reg:squarederror", use_quantile_alpha=False)
        return {
            "name": "xgboost_squarederror",
            "model": models,
            "pred_train": pred_train,
            "pred_val": pred_val,
            "pred_test": pred_test,
        }
    except Exception as square_exc:
        print("[SKIP] XGBoost failed completely:", repr(square_exc))
        return None

## 9. Result saving and leaderboard utilities

The notebook writes results after each fitted model. This makes the run robust: if an optional model fails or takes too long, already completed baseline results are preserved.

Output files:

```text
metrics.csv
leaderboard_test.csv
predictions_val_test.parquet
run_config.json
feature_columns_used.json
models/*.joblib
```

In [11]:

def save_model_object(model_object: Any, model_name: str, paths: ExperimentPaths, cfg: ExperimentConfig) -> None:
    """Save a fitted model object if joblib is available and saving is enabled."""
    if not cfg.save_models or joblib_dump is None:
        return

    try:
        joblib_dump(model_object, paths.models_dir / f"{model_name}.joblib")
    except Exception as exc:
        print(f"[WARN] Could not save model {model_name}: {exc}")


def add_metric_rows_for_model(
    rows: List[Dict[str, Any]],
    prediction_frames: List[pd.DataFrame],
    bundle: DatasetBundle,
    result: Mapping[str, Any],
    cfg: ExperimentConfig,
) -> None:
    """Evaluate a fitted model on train/val/test and append artifacts."""
    model_name = str(result["name"])

    split_payloads = {
        "train": (bundle.train_df, bundle.y_train, result["pred_train"], bundle.w_train),
        "val": (bundle.val_df, bundle.y_val, result["pred_val"], bundle.w_val),
        "test": (bundle.test_df, bundle.y_test, result["pred_test"], bundle.w_test),
    }

    postprocessed_predictions: Dict[str, np.ndarray] = {}

    for split_name, (split_df, y_true, pred_raw, weights) in split_payloads.items():
        pred_raw = np.asarray(pred_raw, dtype=np.float64)
        pred = postprocess_predictions(pred_raw, cfg.postprocess)
        postprocessed_predictions[split_name] = pred

        metrics = evaluate_predictions(split_df, y_true, pred, pred_raw, weights)
        metrics.update(
            {
                "model": model_name,
                "variant": "postprocessed",
                "split": split_name,
                "postprocess": cfg.postprocess,
            }
        )
        rows.append(metrics)

        # Save val/test predictions only to keep the artifact compact.
        if split_name in {"val", "test"}:
            prediction_frames.append(
                prediction_frame(
                    split_df=split_df,
                    model_name=model_name,
                    split_name=split_name,
                    y_pred=pred,
                    y_pred_raw=pred_raw,
                    label_columns=bundle.label_columns,
                    variant="postprocessed",
                )
            )

    if cfg.calibrate:
        correction = validation_median_calibration(bundle.y_val, postprocessed_predictions["val"])

        for split_name in ["val", "test"]:
            split_df, y_true, _, weights = split_payloads[split_name]
            pred_base = postprocessed_predictions[split_name]
            pred_calibrated = apply_calibration(pred_base, correction, cfg.postprocess)

            metrics = evaluate_predictions(split_df, y_true, pred_calibrated, pred_base, weights)
            metrics.update(
                {
                    "model": model_name,
                    "variant": "val_median_calibrated",
                    "split": split_name,
                    "postprocess": cfg.postprocess,
                    "cal_correction_q25": float(correction[0]),
                    "cal_correction_q50": float(correction[1]),
                    "cal_correction_q75": float(correction[2]),
                }
            )
            rows.append(metrics)

            prediction_frames.append(
                prediction_frame(
                    split_df=split_df,
                    model_name=model_name,
                    split_name=split_name,
                    y_pred=pred_calibrated,
                    y_pred_raw=pred_base,
                    label_columns=bundle.label_columns,
                    variant="val_median_calibrated",
                )
            )


def build_leaderboard(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Build a compact test leaderboard sorted by pinball loss."""
    test = metrics_df[metrics_df["split"].eq("test")].copy()
    if test.empty:
        return test

    columns = [
        "model",
        "variant",
        "n_rows",
        "pinball_mean",
        "mae_q50",
        "mae_q75",
        "rmse_q50",
        "iqr_mae",
        "stress_top10_mae_q75",
        "sparse_bottom25_mae_q75",
        "raw_crossing_rate",
    ]
    columns = [col for col in columns if col in test.columns]

    leaderboard = test[columns].sort_values(["pinball_mean", "mae_q75", "mae_q50"], ascending=True)
    leaderboard.insert(0, "rank_by_test_pinball", np.arange(1, len(leaderboard) + 1))
    return leaderboard


def write_static_run_artifacts(paths: ExperimentPaths, cfg: ExperimentConfig, bundle: DatasetBundle) -> None:
    """Write run configuration and the final feature list used by the notebook."""
    run_config = {
        "notebook": "FREIGHT-MNet M0 Baselines Notebook",
        "created_at_unix": time.time(),
        "config": {key: str(value) if isinstance(value, Path) else value for key, value in asdict(cfg).items()},
        "paths": {
            "data_root": str(paths.data_root),
            "scope": paths.scope,
            "supervised_path": str(paths.supervised_path),
            "manifest_path": str(paths.manifest_path),
            "output_dir": str(paths.output_dir),
        },
        "dataset": {
            "n_rows": int(len(bundle.df)),
            "n_features": int(len(bundle.feature_columns)),
            "label_columns": bundle.label_columns,
            "split_counts": {str(k): int(v) for k, v in bundle.df["split"].value_counts().to_dict().items()},
            "year_counts": {str(k): int(v) for k, v in bundle.df["year"].value_counts().sort_index().to_dict().items()},
        },
        "package_versions": {
            "python": sys.version.replace("\n", " "),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "sklearn": sklearn.__version__,
        },
    }

    (paths.output_dir / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")
    (paths.output_dir / "feature_columns_used.json").write_text(
        json.dumps({"feature_columns": bundle.feature_columns}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def flush_current_results(
    rows: List[Dict[str, Any]],
    prediction_frames: List[pd.DataFrame],
    paths: ExperimentPaths,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Write metrics, leaderboard, and prediction artifacts to disk."""
    metrics_df = pd.DataFrame(rows)
    leaderboard = build_leaderboard(metrics_df)

    metrics_df.to_csv(paths.output_dir / "metrics.csv", index=False, encoding="utf-8-sig")
    leaderboard.to_csv(paths.output_dir / "leaderboard_test.csv", index=False, encoding="utf-8-sig")

    if prediction_frames:
        pd.concat(prediction_frames, axis=0, ignore_index=True).to_parquet(
            paths.output_dir / "predictions_val_test.parquet",
            index=False,
        )

    return metrics_df, leaderboard

## 10. Run all M0 baselines

Run this cell to train and evaluate the baseline suite.

Recommended first run:

```python
CONFIG.skip_xgboost = True
```

After the first run succeeds, you can rerun the notebook with `skip_xgboost=False` if your XGBoost version supports the needed objective.

In [12]:

def run_m0_baselines(bundle: DatasetBundle, paths: ExperimentPaths, cfg: ExperimentConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run all configured M0 baselines and return metrics plus leaderboard."""
    np.random.seed(cfg.seed)
    write_static_run_artifacts(paths, cfg, bundle)

    rows: List[Dict[str, Any]] = []
    prediction_frames: List[pd.DataFrame] = []

    fitters = [
        fit_global_train_median,
        fit_od_historical_median,
        fit_ridge,
        fit_elasticnet,
        fit_lightgbm_quantile,
        fit_xgboost,
    ]

    for fitter in fitters:
        start_time = time.time()
        result = fitter(bundle, cfg)

        if result is None:
            continue

        model_name = str(result["name"])
        elapsed = time.time() - start_time
        print(f"\n[DONE] {model_name} fitted in {elapsed:.2f} seconds")

        save_model_object(result.get("model"), model_name, paths, cfg)
        add_metric_rows_for_model(rows, prediction_frames, bundle, result, cfg)

        # Save after each model so partial results are preserved.
        metrics_df, leaderboard = flush_current_results(rows, prediction_frames, paths)

        print("Current test leaderboard:")
        display(leaderboard)

    metrics_df, leaderboard = flush_current_results(rows, prediction_frames, paths)

    print("\n" + "=" * 88)
    print("M0 BASELINES COMPLETE")
    print("=" * 88)
    print("Output directory:", paths.output_dir)
    print("Metrics:        ", paths.output_dir / "metrics.csv")
    print("Leaderboard:    ", paths.output_dir / "leaderboard_test.csv")
    print("Predictions:    ", paths.output_dir / "predictions_val_test.parquet")
    print("=" * 88)

    return metrics_df, leaderboard


METRICS_DF, LEADERBOARD_DF = run_m0_baselines(BUNDLE, PATHS, CONFIG)


[DONE] global_train_median fitted in 0.00 seconds
Current test leaderboard:


,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,rmse_q50,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
4,1,global_train_median,val_median_calibrated,10574,542.233207,989.198023,1490.079161,1185.063773,835.356471,3212.790293,1675.528814,0.0
2,2,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,1185.498184,843.408850,3312.300293,1705.145227,0.0



[DONE] od_historical_median fitted in 10.71 seconds
Current test leaderboard:


,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,rmse_q50,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
7,1,od_historical_median,postprocessed,10574,153.000149,230.534636,472.672178,331.740711,385.553897,709.868526,577.776247,0.0
9,2,od_historical_median,val_median_calibrated,10574,158.227579,244.008809,482.032129,344.457974,386.411862,724.578299,587.356420,0.0
4,3,global_train_median,val_median_calibrated,10574,542.233207,989.198023,1490.079161,1185.063773,835.356471,3212.790293,1675.528814,0.0
2,4,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,1185.498184,843.408850,3312.300293,1705.145227,0.0



[DONE] ridge fitted in 0.33 seconds
Current test leaderboard:


...


[DONE] elasticnet fitted in 174.32 seconds
Current test leaderboard:


,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,rmse_q50,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
7,1,od_historical_median,postprocessed,10574,153.000149,230.534636,472.672178,331.740711,385.553897,709.868526,577.776247,0.000000
9,2,od_historical_median,val_median_calibrated,10574,158.227579,244.008809,482.032129,344.457974,386.411862,724.578299,587.356420,0.000000
17,3,elasticnet,postprocessed,10574,297.492554,486.803579,883.067185,629.019226,632.307697,1899.167730,1016.211197,0.008039
19,4,elasticnet,val_median_calibrated,10574,299.731730,494.900080,890.734596,644.981918,631.964995,1961.823738,1022.312694,0.000000
14,5,ridge,val_median_calibrated,10574,301.979015,497.997692,893.155284,653.726603,630.208209,2040.842019,1035.562082,0.000000
12,6,ridge,postprocessed,10574,302.961809,491.326767,890.884641,640.451216,635.011288,2025.199737,1033.422572,0.007187
4,7,global_train_median,val_median_calibrated,10574,542.233207,989.198023,1490.079161,1185.063773,835.356471,3212.790293,1675.528814,0.000000
2,8,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,1185.498184,843.408850,3312.300293,1705.145227,0.000000


[LightGBM] Training quantile tau=0.25 for target=truck_q25
[LightGBM] Training quantile tau=0.50 for target=truck_q50
[LightGBM] Training quantile tau=0.75 for target=truck_q75

[DONE] lightgbm_quantile fitted in 173.44 seconds
Current test leaderboard:


,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,rmse_q50,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
7,1,od_historical_median,postprocessed,10574,153.000149,230.534636,472.672178,331.740711,385.553897,709.868526,577.776247,0.000000
22,2,lightgbm_quantile,postprocessed,10574,157.827689,281.569916,575.876020,396.373108,524.510670,687.376490,690.101260,0.000757
9,3,od_historical_median,val_median_calibrated,10574,158.227579,244.008809,482.032129,344.457974,386.411862,724.578299,587.356420,0.000000
24,4,lightgbm_quantile,val_median_calibrated,10574,181.692244,293.130485,592.208546,410.568103,485.398813,826.679005,699.007694,0.000000
17,5,elasticnet,postprocessed,10574,297.492554,486.803579,883.067185,629.019226,632.307697,1899.167730,1016.211197,0.008039
19,6,elasticnet,val_median_calibrated,10574,299.731730,494.900080,890.734596,644.981918,631.964995,1961.823738,1022.312694,0.000000
14,7,ridge,val_median_calibrated,10574,301.979015,497.997692,893.155284,653.726603,630.208209,2040.842019,1035.562082,0.000000
12,8,ridge,postprocessed,10574,302.961809,491.326767,890.884641,640.451216,635.011288,2025.199737,1033.422572,0.007187
4,9,global_train_median,val_median_calibrated,10574,542.233207,989.198023,1490.079161,1185.063773,835.356471,3212.790293,1675.528814,0.000000
2,10,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,1185.498184,843.408850,3312.300293,1705.145227,0.000000


[SKIP] XGBoost disabled by config.skip_xgboost=True

M0 BASELINES COMPLETE
Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf
Metrics:         E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\metrics.csv
Leaderboard:     E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv
Predictions:     E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\predictions_val_test.parquet


## 11. Inspect final outputs

Use this cell after training to inspect the final leaderboard and confirm output files were created.

In [13]:

print("Final test leaderboard sorted by pinball_mean:")
display(LEADERBOARD_DF)

print("\nMetrics shape:", METRICS_DF.shape)
print("Saved files:")
for path in sorted(PATHS.output_dir.glob("*")):
    print("  -", path)

Final test leaderboard sorted by pinball_mean:


,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,rmse_q50,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
7,1,od_historical_median,postprocessed,10574,153.000149,230.534636,472.672178,331.740711,385.553897,709.868526,577.776247,0.000000
22,2,lightgbm_quantile,postprocessed,10574,157.827689,281.569916,575.876020,396.373108,524.510670,687.376490,690.101260,0.000757
9,3,od_historical_median,val_median_calibrated,10574,158.227579,244.008809,482.032129,344.457974,386.411862,724.578299,587.356420,0.000000
24,4,lightgbm_quantile,val_median_calibrated,10574,181.692244,293.130485,592.208546,410.568103,485.398813,826.679005,699.007694,0.000000
17,5,elasticnet,postprocessed,10574,297.492554,486.803579,883.067185,629.019226,632.307697,1899.167730,1016.211197,0.008039
19,6,elasticnet,val_median_calibrated,10574,299.731730,494.900080,890.734596,644.981918,631.964995,1961.823738,1022.312694,0.000000
14,7,ridge,val_median_calibrated,10574,301.979015,497.997692,893.155284,653.726603,630.208209,2040.842019,1035.562082,0.000000
12,8,ridge,postprocessed,10574,302.961809,491.326767,890.884641,640.451216,635.011288,2025.199737,1033.422572,0.007187
4,9,global_train_median,val_median_calibrated,10574,542.233207,989.198023,1490.079161,1185.063773,835.356471,3212.790293,1675.528814,0.000000
2,10,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,1185.498184,843.408850,3312.300293,1705.145227,0.000000



Metrics shape: (25, 41)
Saved files:
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\feature_columns_used.json
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\metrics.csv
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\models
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\predictions_val_test.parquet
  - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\run_config.json


## 12. Optional: inspect predictions for the best test model

This cell loads the saved validation/test prediction artifact and shows rows from the best-ranked test model.

In [14]:

prediction_path = PATHS.output_dir / "predictions_val_test.parquet"

if prediction_path.exists() and not LEADERBOARD_DF.empty:
    predictions = pd.read_parquet(prediction_path)
    best_model = LEADERBOARD_DF.iloc[0]["model"]
    best_variant = LEADERBOARD_DF.iloc[0]["variant"]

    print("Best model:", best_model)
    print("Best variant:", best_variant)

    best_predictions = predictions[
        predictions["model"].eq(best_model)
        & predictions["variant"].eq(best_variant)
        & predictions["eval_split"].eq("test")
    ].copy()

    display(best_predictions.head(20))
else:
    print("Prediction artifact not found or leaderboard is empty.")

Best model: od_historical_median
Best variant: postprocessed


,model,variant,eval_split,scope,faf_orig,faf_dest,faf_od,year,split,n_fmi_county_pairs,...,actual_truck_q50,pred_truck_q50,pred_raw_truck_q50,abs_err_truck_q50,actual_truck_q75,pred_truck_q75,pred_raw_truck_q75,abs_err_truck_q75,actual_iqr,pred_iqr
53023,od_historical_median,postprocessed,test,east_plus_gulf,011,011,011->011,2024,test,110,...,60.02,80.420,80.420,20.400,165.00,274.670,274.670,109.670,145.98,251.320
53024,od_historical_median,postprocessed,test,east_plus_gulf,011,012,011->012,2024,test,22,...,860.65,886.820,886.820,26.170,2191.35,1635.630,1635.630,555.720,1968.05,1391.100
53025,od_historical_median,postprocessed,test,east_plus_gulf,011,019,011->019,2024,test,594,...,186.43,199.170,199.170,12.740,774.85,783.030,783.030,8.180,694.37,698.650
53026,od_historical_median,postprocessed,test,east_plus_gulf,011,050,011->050,2024,test,652,...,1200.08,1242.850,1242.850,42.770,1891.48,2055.720,2055.720,164.240,1418.11,1504.990
53027,od_historical_median,postprocessed,test,east_plus_gulf,011,091,011->091,2024,test,16,...,3006.08,2925.000,2925.000,81.080,5393.13,4746.390,4746.390,646.740,3012.30,2628.960
53028,od_historical_median,postprocessed,test,east_plus_gulf,011,092,011->092,2024,test,33,...,2836.18,2700.330,2700.330,135.850,5362.37,4221.830,4221.830,1140.540,3302.27,2332.600
53029,od_historical_median,postprocessed,test,east_plus_gulf,011,099,011->099,2024,test,16,...,3113.83,2949.120,2949.120,164.710,5481.82,4554.900,4554.900,926.920,2938.45,2221.970
53030,od_historical_median,postprocessed,test,east_plus_gulf,011,101,011->101,2024,test,17,...,3202.10,2472.420,2472.420,729.680,5284.07,3704.290,3704.290,1579.780,3140.50,1996.020
53031,od_historical_median,postprocessed,test,east_plus_gulf,011,109,011->109,2024,test,4,...,3161.10,3199.615,3199.615,38.515,5775.17,4934.175,4934.175,840.995,3815.69,2554.875
53032,od_historical_median,postprocessed,test,east_plus_gulf,011,111,011->111,2024,test,3,...,3004.40,2145.905,2145.905,858.495,5326.55,3700.700,3700.700,1625.850,3554.80,2249.100
